# Quadruped Locomotion (Go2): Reward Tuning (보행 자세 개선)

`go2_locomotion_train.ipynb` 에서 학습한 모델을 출발점으로, **보상(reward)을
3번에 걸쳐 수정하고 매번 재학습**하면서 보행 자세가 개선되는 과정을 확인합니다.

각 라운드는 다음 순서로 진행합니다.
1. **현재 모델을 테스트**하고 영상으로 보행을 확인
2. **`envs.yaml`(보상 가중치)을 수정**
3. 수정한 보상으로 **재학습** (Colab 시간 제약상 매번 10,000 step 만 이어학습)

위 1~3 을 **총 3회** 반복하며, 각 라운드의 산출 모델이 다음 라운드의 입력이 됩니다.

> 런타임 → 런타임 유형 변경 → **T4 GPU** 로 설정 후 실행하세요.


---

## 0. 환경 설정

GitHub 레포지토리를 clone 하고 의존성을 설치합니다. (이전 노트북들과 동일)


In [ ]:
# Clone repository
import os, sys

import yaml

# Detect Colab by availability of /content or google.colab.
try:
    import google.colab  # noqa: F401
    in_colab = True
except Exception:
    in_colab = os.path.isdir("/content")

try:
    base_dir
except NameError:
    base_dir = "/content" if in_colab else os.getcwd()
os.chdir(base_dir)

repo_dir = os.path.join(base_dir, "RL_tutorial")

print(f"Base directory: {base_dir}")
print(f"Repo directory: {repo_dir}")

if not os.path.isdir(repo_dir):
  !git clone https://github.com/agbread/RL_tutorial.git
else:
  print("Cloned Directory already exists")

os.chdir(repo_dir)
print("Current Directory: ", os.getcwd())

sys.path.insert(0, os.path.join(repo_dir, "src"))
os.environ["MUJOCO_GL"] = "egl"

# numpy 1.x / 2.x 호환성 패치:
# 저장된 모델이 numpy 2.x (numpy.core_ 경로) 로 직렬화된 경우를 위해
# numpy.core_ 를 numpy.core 의 alias 로 등록합니다.
import numpy, numpy.core, numpy.core.numeric, numpy.core.multiarray
import numpy.random._pickle as _np_pickle

sys.modules['numpy.core_'] = numpy.core
sys.modules['numpy.core_.numeric'] = numpy.core.numeric
sys.modules['numpy.core_.multiarray'] = numpy.core.multiarray

_orig_bg_ctor = _np_pickle.__bit_generator_ctor
def _patched_bg_ctor(bg='MT19937'):
    return bg() if isinstance(bg, type) else _orig_bg_ctor(bg)
_np_pickle.__bit_generator_ctor = _patched_bg_ctor

In [ ]:
# Install dependencies
# stable-baselines3는 PyPI에서 설치합니다 (이 repo에는 sb3 소스 포크가 없음).
# 로컬에서 검증된 버전 조합으로 고정합니다.
!pip install "stable-baselines3==2.3.0" "gymnasium==0.29.1" "mujoco==3.8.0" "numpy<2" "imageio[ffmpeg]" tensorboard pygments

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter
import inspect

def _render_code(code, title="code", max_height=400, bg="transparent", indent=16):
    style_name = "native" if in_colab else "friendly"
    formatter = HtmlFormatter(style=style_name, noclasses=True, linenos="inline")
    html = highlight(code, PythonLexer(), formatter)
    css = """
    <style>
    .highlight pre { margin: 0; text-align: left; }
    </style>
    """
    return HTML(f"""
    {css}
    <details>
      <summary>{title}</summary>
      <div style="margin-top:8px; margin-left:{indent}px; max-height:{max_height}px; overflow:auto; border:1px solid #ddd; padding:10px; background:{bg};">
        {html}
      </div>
    </details>
    """)


def show_code(path, max_height=400, bg="transparent"):
    code = Path(path).read_text()
    return _render_code(code, title=str(path), max_height=max_height, bg=bg)

def show_func(obj, max_height=400, bg="transparent"):
    code = inspect.getsource(obj)
    return _render_code(code, max_height=max_height, bg=bg)

---

## 1. 설정 파일 살펴보기

튜닝의 대상이 되는 보상 설정과 그 구현을 먼저 확인합니다.

- **`src/params.yaml`** — PPO 하이퍼파라미터/학습 설정
- **`src/envs.yaml`** — 보상/패널티 가중치 (이번 노트북에서 라운드마다 수정할 대상)
- **`src/mdp/reward.py`** — 가중치가 실제로 계산되는 보상 함수 구현


In [ ]:
display(show_code(f"{repo_dir}/src/params.yaml"))
display(show_code(f"{repo_dir}/src/envs.yaml"))
display(show_code(f"{repo_dir}/src/mdp/reward.py", max_height=600))

---

## 2. 공통 헬퍼 정의

3개 라운드에서 반복 사용할 함수와 설정을 정의합니다.

- 상단 설정: `INITIAL_MODEL_PATH`(출발 모델), `ADDITIONAL_TIMESTEPS`, `N_ENVS`, `SEED`
- `rollout_and_video(model, cfg, tag)` — 모델을 롤아웃해 mp4 저장 (test 로직 이식)
- `show_video(path)` — 노트북에서 영상 재생
- `make_reward_cfg(updates, out, base)` — 기준 envs.yaml 에서 보상 가중치를 바꿔 저장
- `finetune(model_in, cfg, tag)` — 해당 보상으로 10,000 step 이어학습


In [ ]:
import time, gc, shutil
import numpy as np
import imageio
from tqdm.auto import tqdm

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, CallbackList,
)
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv

import src.go2_mujoco_env as go2_env
from src.utils.reward_logging_callback import RewardLoggingCallback
from IPython.display import Video

# 학습 설정 로드
with open(f"{repo_dir}/src/params.yaml", "r", encoding="utf-8") as f:
    policy_cfg = yaml.safe_load(f)

# ===== 설정 (필요시 수정) =====
# TODO: 1,990,000 step 까지 학습한 실제 모델(이전 노트북 산출물) 경로로 교체.
#       아직 그 모델이 없으므로 지금은 repo 에 있는 모델을 기본값으로 둔다.
INITIAL_MODEL_PATH = f"{repo_dir}/models/Go2_forth_test/best_model.zip"
ADDITIONAL_TIMESTEPS = 10_000   # 매 라운드 추가 학습 step
N_ENVS = policy_cfg["n_envs"]   # Colab 자원에 맞게 줄여도 됨 (예: 4)
SEED = policy_cfg["seed"]
BASE_CFG = f"{repo_dir}/src/envs.yaml"

assert os.path.exists(INITIAL_MODEL_PATH), (
    f"출발 모델을 찾을 수 없습니다: {INITIAL_MODEL_PATH}\n"
    f"INITIAL_MODEL_PATH 를 존재하는 .zip 경로로 수정하세요."
)

VIDEO_DIR = f"{repo_dir}/models/_reward_tuning_videos"
os.makedirs(VIDEO_DIR, exist_ok=True)


def rollout_and_video(model_path, env_cfg_path, tag):
    """모델을 롤아웃하여 mp4 로 저장하고 경로를 반환."""
    env = go2_env.Go2MujocoEnv(
        prj_path=repo_dir, cfg_path=env_cfg_path,
        given_command=[0.9, 0.0, 0.0],
        render_mode="rgb_array", camera_name="tracking",
        width=960, height=540,
    )
    env._reset_noise_scale = 0.05
    custom_objects = {
        "observation_space": env.observation_space,
        "action_space": env.action_space,
    }
    model = PPO.load(model_path, env=env, custom_objects=custom_objects, verbose=0)

    video_fps = 10
    render_interval = 50 // video_fps               # control 50Hz
    max_steps = int(policy_cfg["test"]["max_time_step_s"] * 50)
    video_path = f"{VIDEO_DIR}/rollout_{tag}.mp4"

    obs, _ = env.reset()
    frames = []
    for step in tqdm(range(max_steps), desc=f"rollout[{tag}]", unit="step"):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        if step % render_interval == 0:
            frames.append(env.render())
        if terminated or truncated:
            obs, _ = env.reset()
    env.close()

    imageio.mimwrite(video_path, frames, fps=video_fps,
                     codec="libx264", quality=8, pixelformat="yuv420p")
    print("saved video:", video_path)
    return video_path


def show_video(video_path):
    display(Video(video_path, embed=True, html_attributes="controls autoplay loop"))


def make_reward_cfg(updates, out_path, base_cfg_path):
    """base_cfg_path 의 envs.yaml 을 읽어 updates(점 표기 키)를 덮어쓴 뒤 저장."""
    with open(base_cfg_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    print(f"reward 변경 ({os.path.basename(base_cfg_path)} 기준):")
    for dotted, val in updates.items():
        section, key = dotted.split(".")
        old = cfg.get(section, {}).get(key)
        cfg[section][key] = val
        print(f"  {dotted}: {old} -> {val}")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
    print("saved reward cfg:", out_path)
    return out_path


def finetune(model_in_path, env_cfg_path, run_tag):
    """env_cfg_path 보상으로 model_in_path 를 ADDITIONAL_TIMESTEPS 만큼 이어학습."""
    log_dir = f"{repo_dir}/logs"
    os.makedirs(log_dir, exist_ok=True)
    env_kwargs = {"prj_path": repo_dir, "cfg_path": env_cfg_path}
    vec_env = make_vec_env(go2_env.Go2MujocoEnv, env_kwargs=env_kwargs,
                           n_envs=N_ENVS, seed=SEED, vec_env_cls=SubprocVecEnv)
    eval_env = make_vec_env(go2_env.Go2MujocoEnv, env_kwargs=env_kwargs,
                            n_envs=1, seed=SEED + 10_000, vec_env_cls=DummyVecEnv)

    run_name = time.strftime("%Y-%m-%d_%H-%M-%S") + f"-{run_tag}"
    model_path = f"{repo_dir}/models/{run_name}"
    os.makedirs(model_path, exist_ok=True)
    shutil.copy2(env_cfg_path, f"{model_path}/envs.yaml")   # 사용한 보상 설정 보관
    print("저장 위치:", model_path)

    _dummy = go2_env.Go2MujocoEnv(prj_path=repo_dir, cfg_path=env_cfg_path, render_mode=None)
    custom_objects = {
        "observation_space": _dummy.observation_space,
        "action_space": _dummy.action_space,
    }
    _dummy.close()

    callbacks = CallbackList([
        EvalCallback(eval_env, best_model_save_path=model_path, log_path=log_dir,
                     eval_freq=max(policy_cfg["eval_freq"] // N_ENVS, 1),
                     n_eval_episodes=5, deterministic=True, render=False),
        CheckpointCallback(
            save_freq=max(policy_cfg["policy"]["n_steps"] * policy_cfg["log"]["interval"] // N_ENVS, 1),
            save_path=model_path, name_prefix="model",
            save_replay_buffer=False, save_vecnormalize=False),
        RewardLoggingCallback(),
    ])

    print(f"[{run_tag}] load pretrained: {model_in_path}")
    model = PPO.load(model_in_path, env=vec_env, custom_objects=custom_objects,
                     verbose=1, tensorboard_log=log_dir)
    model.learning_rate = policy_cfg["policy"]["learning_rate"]
    model._setup_lr_schedule()
    model.learn(total_timesteps=ADDITIONAL_TIMESTEPS, reset_num_timesteps=False,
                progress_bar=True, tb_log_name=run_name, callback=callbacks)
    model.save(f"{model_path}/final_model")

    vec_env.close(); eval_env.close()
    del model; gc.collect()

    out = f"{model_path}/best_model.zip"
    if not os.path.exists(out):
        out = f"{model_path}/final_model.zip"
    print(f"[{run_tag}] saved ->", out)
    return out


# 라운드 체인 상태: 현재 모델 / 현재 보상 설정 / 비교용 영상 목록
cur_model = INITIAL_MODEL_PATH
cur_cfg = BASE_CFG
round_videos = []
print("출발 모델:", cur_model)

---

## 3. 라운드 1 — 발 들어올림(Foot Clearance) 강화

### (a) 현재 모델 보행 확인

이번 라운드의 입력 모델을 먼저 롤아웃해 **현재 보행 상태를 영상으로 확인**합니다.


In [ ]:
vp = rollout_and_video(cur_model, cur_cfg, tag="0_initial")
round_videos.append((vp, "초기 모델"))
show_video(vp)

### (b) reward 수정 후 재학습

발이 지면을 끄는 현상을 줄이기 위해 **발 높이 패널티 `cost.foot_clearance`(→ 40)** 와 **체공 시간 보상 `reward.feet_air_time`(→ 0.5)** 을 키웁니다. 발을 더 또렷하게 들어올리는 보행을 유도합니다.

수정한 보상으로 입력 모델을 **10,000 step 이어학습**하고,
산출 모델/보상 설정을 다음 라운드 입력으로 넘깁니다.

> 실제 변경 전/후 값은 아래 셀 출력(`make_reward_cfg`)에 표시됩니다.

In [ ]:
updates = {"cost.foot_clearance": 40.0, "reward.feet_air_time": 0.5}
cfg_tuned = make_reward_cfg(
    updates, out_path=f"{repo_dir}/models/_reward_cfgs/envs_tune1.yaml",
    base_cfg_path=cur_cfg,
)
new_model = finetune(cur_model, cfg_tuned, run_tag="tune1")

# 다음 라운드로 체인
cur_model = new_model
cur_cfg = cfg_tuned

---

## 4. 라운드 2 — Trot 보행 규칙성 강화

### (a) 현재 모델 보행 확인

이번 라운드의 입력 모델을 먼저 롤아웃해 **현재 보행 상태를 영상으로 확인**합니다.


In [ ]:
vp = rollout_and_video(cur_model, cur_cfg, tag="1_tune1")
round_videos.append((vp, "튜닝1 후 모델"))
show_video(vp)

### (b) reward 수정 후 재학습

대각선 다리쌍이 함께 움직이는 trot 패턴을 더 강하게 따르도록 **`cost.gait_enforcement`(→ 0.08)** 를 키웁니다. 보행 리듬이 불규칙한 문제를 개선합니다.

수정한 보상으로 입력 모델을 **10,000 step 이어학습**하고,
산출 모델/보상 설정을 다음 라운드 입력으로 넘깁니다.

> 실제 변경 전/후 값은 아래 셀 출력(`make_reward_cfg`)에 표시됩니다.

In [ ]:
updates = {"cost.gait_enforcement": 0.08}
cfg_tuned = make_reward_cfg(
    updates, out_path=f"{repo_dir}/models/_reward_cfgs/envs_tune2.yaml",
    base_cfg_path=cur_cfg,
)
new_model = finetune(cur_model, cfg_tuned, run_tag="tune2")

# 다음 라운드로 체인
cur_model = new_model
cur_cfg = cfg_tuned

---

## 5. 라운드 3 — 자세 안정화 (Hip Spread / 관절 자세)

### (a) 현재 모델 보행 확인

이번 라운드의 입력 모델을 먼저 롤아웃해 **현재 보행 상태를 영상으로 확인**합니다.


In [ ]:
vp = rollout_and_video(cur_model, cur_cfg, tag="2_tune2")
round_videos.append((vp, "튜닝2 후 모델"))
show_video(vp)

### (b) reward 수정 후 재학습

다리가 바깥으로 벌어지거나 기본 자세에서 크게 벗어나는 것을 막기 위해 **`cost.hip_spread`(→ 0.3)** 와 **`cost.joint_pos_deviation`(→ 0.12)** 를 키웁니다. 더 곧고 안정적인 자세를 유도합니다.

수정한 보상으로 입력 모델을 **10,000 step 이어학습**하고,
산출 모델/보상 설정을 다음 라운드 입력으로 넘깁니다.

> 실제 변경 전/후 값은 아래 셀 출력(`make_reward_cfg`)에 표시됩니다.

In [ ]:
updates = {"cost.hip_spread": 0.3, "cost.joint_pos_deviation": 0.12}
cfg_tuned = make_reward_cfg(
    updates, out_path=f"{repo_dir}/models/_reward_cfgs/envs_tune3.yaml",
    base_cfg_path=cur_cfg,
)
new_model = finetune(cur_model, cfg_tuned, run_tag="tune3")

# 다음 라운드로 체인
cur_model = new_model
cur_cfg = cfg_tuned

---

## 6. 최종 결과 확인

라운드 3 까지 튜닝한 최종 모델의 보행을 확인합니다.


In [ ]:
vp = rollout_and_video(cur_model, cur_cfg, tag="3_tune3_final")
round_videos.append((vp, "튜닝3 후 모델(최종)"))
show_video(vp)

### 라운드별 보행 비교

초기 → 튜닝1 → 튜닝2 → 튜닝3(최종) 의 보행 영상을 나란히 비교합니다.


In [ ]:
import base64

def _embed(path, caption, width=320):
    b64 = base64.b64encode(open(path, "rb").read()).decode("ascii")
    return f"""
    <figure style="margin:0; text-align:center;">
      <video controls autoplay muted loop width="{width}">
        <source src="data:video/mp4;base64,{b64}" type="video/mp4">
      </video>
      <figcaption style="margin-top:6px; font-size:12px;">{caption}</figcaption>
    </figure>
    """

display(HTML(
    '<div style="display:flex; gap:12px; flex-wrap:wrap;">'
    + "".join(_embed(p, c) for p, c in round_videos if os.path.exists(p))
    + "</div>"
))